# 🤖 02 — BERT Mimarisi: İstanbul için Dil Modeli

Bu notebook BERT mimarisini açıklar ve İstanbul soruları üzerinde embedding analizleri yapar.

In [ ]:
!pip install transformers torch scikit-learn matplotlib seaborn -q
print("✅ Hazır!")

## 🏗️ BERT Mimarisi



In [ ]:
import torch
import numpy as np
from transformers import BertTokenizer, BertModel
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

tokenizer = BertTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")
model = BertModel.from_pretrained("dbmdz/bert-base-turkish-cased")
model.eval()
print("✅ BERT Turkish modeli yüklendi!")

## 🔤 Adım 1: Tokenizasyon

In [ ]:
sorular = [
    "Ayasofya nerede?",
    "Topkapı Sarayı ne zaman yapıldı?",
    "İstanbul metrosu nasıl kullanılır?",
    "Boğaz ne kadar uzun?",
]

for soru in sorular:
    tokens = tokenizer.tokenize(soru)
    ids = tokenizer.encode(soru)
    print(f"
📝 Girdi : {soru}")
    print(f"   Token : {tokens}")
    print(f"   ID    : {ids}")

## 📐 Adım 2: Embedding Uzayında İstanbul Soruları

In [ ]:
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=64)
    with torch.no_grad():
        output = model(**inputs)
    return output.last_hidden_state[:, 0, :].numpy()

sorular_extended = [
    "Ayasofya kaç yılında yapıldı?",
    "Galata Kulesi nerede?",
    "İstanbul metrosu var mı?",
    "Vapur nerede kalkar?",
    "Türk kahvaltısı ne içerir?",
    "Kadıköy nerede?",
    "Boğaz köprüleri hangileri?",
    "Topkapı Sarayı kaç yıl kullanıldı?",
]

vectors = np.vstack([get_embedding(s) for s in sorular_extended])
print(f"✅ Embedding boyutu: {vectors.shape}")

## 📊 Adım 3: PCA ile 2D Görselleştirme

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#e74c3c","#e74c3c","#3498db","#3498db","#2ecc71","#f39c12","#3498db","#e74c3c"]
ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=120, zorder=5)
for i, soru in enumerate(sorular_extended):
    ax.annotate(soru[:25], (coords[i,0], coords[i,1]), fontsize=8, ha="center", va="bottom")
ax.set_title("🗺️ İstanbul Soruları — BERT Embedding Uzayı (PCA 2D)")
ax.set_xlabel("PCA Bileşen 1")
ax.set_ylabel("PCA Bileşen 2")
plt.tight_layout()
plt.savefig("bert_embedding_pca.png", dpi=120)
plt.show()

## 🔗 Adım 4: Kosinüs Benzerlik Haritası

In [ ]:
sim_matrix = cosine_similarity(vectors)

fig, ax = plt.subplots(figsize=(10, 8))
kisalt = [s[:20]+"..." for s in sorular_extended]
sns.heatmap(sim_matrix, annot=True, fmt=".2f", xticklabels=kisalt, yticklabels=kisalt, cmap="RdYlGn", ax=ax)
ax.set_title("🌡️ Sorular Arası Kosinüs Benzerlik Matrisi")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("cosine_similarity.png", dpi=120)
plt.show()
print("✅ Benzerlik matrisi tamamlandı!")